In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import getpass

password = getpass.getpass("Enter MySQL password: ")
user = "root"
host = "localhost"
db = "sakila"

connection_string = f"mysql+pymysql://{user}:{password}@{host}/{db}"
engine = create_engine(connection_string)

with engine.connecction() as conn:
    print("Connection successfully!")

def rentals_month(engine, month, year):
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month}
            AND YEAR(rental_date) = {year}
    """
    df = pd.read_sql(query, engine)
    return df

def rental_count_month(df, month, year):
    col_name = f"rentals_{month:02d}_{year}"

    result = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=col_name)
    )
    return result

def compare_rentals(df1, df2):
    combined = pd.merge(df1, df2, on="customer_id", how="outer")
    combined = combined.fillna(0)

    rental_cols = [c for c in combined.columns if c != "customer_id"]
    combined["difference"] = combined[rental_cols[0]] - combined[rental_cols[1]]

    return combined

may_df = rentals_month(engine, 5, 2005)
june_df = rentals_month(engine, 6, 2005)

may_counts = rental_count_month(may_df, 5, 2005)
june_counts = rental_count_month(june_df, 6,2005)

comparison = compare_rentals(may_counts, june_counts)
comparison.head()